# 22차시 실습 — 내 서비스를 개선·최적화

> **day1 연속 실습.** day1·7차시에서 만든 **맛집 추천 도우미**(내 서비스의 AI 기능)를 이제 **운영 품질**로 끌어올립니다.
> 오늘은 ① **개선 1사이클**(로그→가설→수정→재평가, 전/후 점수) ② **캐싱**(반복 호출 지연↓) ③ **모델 라우팅**(난이도별 싼/강한 모델)을 직접 돌려봅니다.

## 🧪 사용법
- 워크샵 본 실습은 **Codex CLI**로, 이 노트북은 같은 개념을 **OpenAI API로 즉시 실행**해 보는 동반 자료입니다.
- 각 단계에 **⌨️ 터미널 Codex** 명령(복붙용)을 함께 적어 두었습니다.
- 위에서부터 **순서대로** 실행하세요 (`Shift+Enter`).

In [ ]:
# ✅ 환경 준비 — 맨 처음 한 번만 (day1 노트북과 동일)
%pip install -q openai
import os, json, getpass
from openai import OpenAI
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY (화면에 안 보임): ")
client = OpenAI(); MODEL = "gpt-4o-mini"
print("준비 완료 ·", MODEL)

## 오늘의 연속 시나리오

day1·7차시: **맛집 추천 도우미**를 만들었습니다. 그 안에는 사용자 리뷰의 감정을 분류하는 **AI 기능**이 있습니다.

오늘 18차시 트레이스/로그를 보니 이 기능이 가끔 **장황하게 답해서 틀린 것**으로 기록됩니다.
→ 이 **내 서비스의 AI 기능을 개선하고, 호출을 최적화**합니다. 구조는 동일하니 **당신 팀 MVP의 AI 기능**에 그대로 옮길 수 있습니다(STEP 4).

## STEP 1 — 개선 1사이클 (로그→가설→수정→재평가)

18차시의 **트레이스·로그**가 개선의 입력입니다. 로그에서 모은 **실패 사례(평가셋)** 로,
지금 프롬프트(**baseline**)의 점수를 먼저 잰 뒤, 다음 셀에서 프롬프트를 고쳐 **재평가**합니다.

오늘 과제 = 내 맛집 앱의 **리뷰 감정 분류** 기능: 리뷰를 **`긍정`/`부정`/`중립` 한 단어로만** 분류하기.

⌨️ **터미널 Codex:** `codex "리뷰 감정 분류 기능을 평가셋으로 채점하는 코드를 만들어줘"`

In [ ]:
# 로그에서 모은 평가셋 (리뷰, 정답 라벨) — 18차시 트레이스가 이 데이터의 출처
EVAL_SET = [
    ("이 집 갈비찜 정말 최고였어요, 또 가고 싶어요!", "긍정"),
    ("최악의 서비스. 다시는 안 갑니다.", "부정"),
    ("그냥 그래요. 특별할 건 없네요.", "중립"),
    ("양 많고 가격도 만족스럽습니다.", "긍정"),
    ("기대했는데 너무 실망했어요.", "부정"),
    ("가격 대비 무난한 편이에요.", "중립"),
]

def evaluate(system_prompt, label, verbose=True):
    """평가셋 전체를 돌려 '정답 라벨과 정확히 일치'한 비율을 반환."""
    correct = 0
    for text, gold in EVAL_SET:
        resp = client.chat.completions.create(model=MODEL, messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": text},
        ])
        out = resp.choices[0].message.content.strip()
        ok = (out == gold)            # 한 단어 정확 일치만 정답
        correct += ok
        if verbose:
            print(f"  {'✅' if ok else '❌'} 기대={gold:<2} | 응답={out!r}")
    score = correct / len(EVAL_SET)
    print(f"[{label}] 점수: {score:.0%} ({correct}/{len(EVAL_SET)})\n")
    return score

# baseline — 막연한 프롬프트 (로그에서 '문장으로 장황하게 답한다'는 실패가 관찰됨)
BASELINE = "리뷰의 감정을 알려줘."
before = evaluate(BASELINE, "baseline")

### 가설 → 수정

**가설**: baseline 점수가 낮은 건 모델이 문장으로 장황하게 답해서 `긍정` 같은 한 단어와 안 맞기 때문.
**수정**: 출력 형식을 **한 단어로 못박고** 선택지를 명시한다 (프롬프트 한 줄 개선).

⌨️ **터미널 Codex:** `codex "감정 분류 프롬프트를 긍정/부정/중립 한 단어만 출력하도록 고쳐줘"`

In [ ]:
# improved — 형식을 못박은 프롬프트
IMPROVED = (
    "너는 맛집 리뷰 감정 분류기다. 입력 리뷰를 보고 "
    "'긍정', '부정', '중립' 중 정확히 한 단어로만 답하라. 다른 말은 절대 붙이지 마라."
)
after = evaluate(IMPROVED, "improved")

print("=" * 32)
print(f"개선 전: {before:.0%}  →  개선 후: {after:.0%}  (Δ {after - before:+.0%})")
print("닫힌 루프: 로그→가설→수정→재평가 1사이클 완료 ✅")

## STEP 2 — 캐싱 (같은 일을 두 번 하지 마라)

인기 지역 추천처럼 **같은 요청이 반복**되면 모델을 다시 부르지 말고 **저장된 답을 재사용**합니다.
프롬프트를 해시(key)로 쓰는 **인메모리 캐시**를 붙여 반복 호출의 **지연·비용**이 얼마나 줄어드는지 봅니다.

⌨️ **터미널 Codex:** `codex "프롬프트 해시를 키로 쓰는 인메모리 캐시로 모델 호출을 감싸줘"`

In [ ]:
import time, hashlib

CACHE = {}            # 프롬프트 해시 → 응답
STATS = {"hit": 0, "miss": 0}

def ask_cached(prompt, model=MODEL):
    key = hashlib.sha256((model + "|" + prompt).encode()).hexdigest()
    t0 = time.time()
    if key in CACHE:                       # 캐시 HIT — 모델 호출 건너뜀
        STATS["hit"] += 1
        return CACHE[key], time.time() - t0, "HIT"
    resp = client.chat.completions.create(model=model, messages=[
        {"role": "user", "content": prompt}])
    out = resp.choices[0].message.content.strip()
    CACHE[key] = out                       # 캐시에 저장
    STATS["miss"] += 1
    return out, time.time() - t0, "MISS"

# 같은 추천 요청을 두 번 (+ 다른 요청 하나) — HIT/MISS와 지연을 비교
for q in ["강남 점심 맛집 한 곳 추천해줘.",
          "강남 점심 맛집 한 곳 추천해줘.",
          "홍대 저녁 맛집 한 곳 추천해줘."]:
    ans, dt, status = ask_cached(q)
    print(f"[{status}] {dt*1000:6.1f} ms | {q}")
print("\n캐시 통계:", STATS)

## STEP 3 — Model routing (작업에 맞는 모델)

"강남 맛집?" 같은 쉬운 요청은 **싸고 빠른 모델**로, "동선·예산·취향 다 고려해 코스 짜줘" 같은 복잡한 요청만 **강한 추론 모델**로 보냅니다.
여기서는 간단한 **휴리스틱 라우터**(길이·키워드)로 모델을 고릅니다.

⌨️ **터미널 Codex:** `codex "질문 난이도를 휴리스틱으로 판단해 싼/강한 모델로 라우팅하는 함수를 만들어줘"`

In [ ]:
CHEAP, STRONG = "gpt-4o-mini", "gpt-4o"   # 워크샵에선 둘 다 mini로 둬도 됨
HARD_HINTS = ("코스", "동선", "예산", "비교", "분석", "왜", "추천 이유", "단계별")

def route(question):
    """어려우면 STRONG, 아니면 CHEAP. (길이 + 키워드 휴리스틱)"""
    hard = len(question) > 40 or any(h in question for h in HARD_HINTS)
    return STRONG if hard else CHEAP

for q in [
    "강남 맛집 한 곳만.",
    "홍대 카페 추천해줘.",
    "2명 예산 4만원으로 강남에서 저녁 코스를 동선까지 고려해 단계별로 짜고 왜 그런지 분석해줘",
]:
    chosen = route(q)
    tier = "강함" if chosen == STRONG else "저렴"
    print(f"[{tier:>2}·{chosen}] {q}")

## 도전 과제 — 캐시 절감 측정

캐시 적중으로 아낀 **호출 수**와 **비용**을 직접 계산해 봅니다.
(아래 셀을 실행하기 전에 STEP 2 캐시 셀을 여러 번 돌려 HIT를 쌓아보세요.)

In [ ]:
COST_PER_CALL = 0.0005   # 호출당 가정 비용($) — 실제 값으로 바꿔보세요

total = STATS["hit"] + STATS["miss"]
hit_rate = STATS["hit"] / total if total else 0
saved = STATS["hit"] * COST_PER_CALL
print(f"총 요청 {total}건 · HIT {STATS['hit']} · MISS {STATS['miss']}")
print(f"캐시 적중률: {hit_rate:.0%}")
print(f"아낀 모델 호출: {STATS['hit']}회 · 절감 비용: ${saved:.4f}")
# 도전: route()로 싼 모델에 보낸 비율까지 더해 총 절감을 추정해보기

## STEP 4 — Retry / Fallback: 실패를 흡수한다

강의의 견고함 — API는 반드시 실패합니다. **지수 백오프 재시도 + 폴백**으로 사용자가 빈 화면을 보지 않게 합니다.

In [ ]:
import time, random

def flaky_call(q):                         # 60% 확률로 실패하는 가짜 API
    if random.random() < 0.6: raise ConnectionError("일시적 API 오류")
    return f"'{q}' 추천 결과입니다."

def robust_call(q, max_retries=3):
    for attempt in range(max_retries):
        try:
            return flaky_call(q), f"성공 (시도 {attempt+1})"
        except ConnectionError:
            wait = 2 ** attempt * 0.1      # 지수 백오프: 0.1 → 0.2 → 0.4초
            print(f"  실패 → {wait:.1f}s 대기 후 재시도")
            time.sleep(wait)
    return "지금은 추천이 어려워요. 잠시 후 다시 시도해 주세요.", "폴백 응답"

random.seed(7)
answer, how = robust_call("강남 매운 점심")
print(f"\n[{how}] {answer}")
# 관찰: 무한 재시도 금지(상한 필수) — 폴백은 다른 모델·캐시 답·안전 기본 응답 중 택1

## STEP 5 — 시맨틱 캐시 맛보기: 비슷한 질문도 적중

STEP 2의 정확 일치 캐시는 한 글자만 달라도 미스 — **유사도 기반**이면 비슷한 질문에도 적중합니다 (오적중 임계값 주의).

In [ ]:
import difflib

SEM_CACHE = {}                                     # {질문: 응답}
def semantic_cached(q, threshold=0.7):
    for cached_q, cached_a in SEM_CACHE.items():
        sim = difflib.SequenceMatcher(None, q, cached_q).ratio()
        if sim >= threshold:
            return cached_a, f"HIT (유사도 {sim:.2f} ← '{cached_q}')"
    r = client.chat.completions.create(model=MODEL,
        messages=[{"role":"user","content":q}])
    a = r.choices[0].message.content
    SEM_CACHE[q] = a
    return a, "MISS → 캐시 저장"

for q in ["강남에서 매운 점심 추천해줘",
          "강남에서 매운 점심 추천해줘!",        # 정확 일치 캐시라면 MISS였을 입력
          "부산 여행 코스 알려줘"]:               # 전혀 다른 질문 — MISS여야 정상
    _a, status = semantic_cached(q)
    print(f"{status:35s} | {q}")
# 관찰: threshold를 0.3으로 낮추면 '부산 여행'도 HIT(오적중!) — 임계값 튜닝이 시맨틱 캐시의 전부
# 실전: 문자열 유사도 대신 임베딩 코사인 유사도를 쓴다 (10차시)

## STEP 6 — ⭐ 내 MVP에 적용

위 3가지(개선 1사이클 · 캐싱 · 라우팅)는 **그대로** 당신 팀 MVP의 AI 기능에 옮길 수 있습니다.
아래 `MY_PLAN_TODO`를 팀 서비스에 맞게 채워보세요.

⌨️ **터미널 Codex:** `codex "우리 MVP AI 기능에 개선 1사이클·캐시·모델 라우팅을 붙여줘"`

In [ ]:
# 팀별 작성: 우리 MVP의 AI 기능을 어떻게 개선·최적화할지 계획
MY_PLAN_TODO = {
    "ai_feature": "",      # 예: "가계부 지출 자동 분류"
    "fail_cases": [],      # 로그에서 모은 실패 입력 몇 개 → 평가셋
    "fix_hypothesis": "",  # 가설: 왜 틀렸나? 무엇을 한 줄 고칠까?
    "cache_target": "",    # 무엇을 캐시할까? (자주 반복되는 요청)
    "route_rule": "",      # 쉬움→싼 모델 / 어려움→강한 모델 기준
}

for k, v in MY_PLAN_TODO.items():
    print(f"{k:<16}: {v or '⬜ 채워주세요'}")
print("\n→ 채운 뒤 STEP 1~3의 evaluate / ask_cached / route 를 우리 기능에 맞게 바꿔 돌리면 '내 MVP 개선·최적화' 완성")

## 정리

- 개선 = **로그→가설→수정→재평가**의 닫힌 루프. 18차시 **트레이스가 연료**, 15차시 평가셋이 채점 기준.
- 스케일 = **캐싱**(중복 연산 회피)·**model routing**(난이도별 모델)·**동시성**·**retry/fallback**로 싸고·빠르고·견고하게.
- 절감도 개선도 **측정 없이는 불가능** — 점수·적중률·지연·비용을 항상 같이 본다.
- ⌨️ 정리: `codex "오늘 붙인 개선 루프·캐시·라우팅을 우리 MVP 레포에 커밋해줘"`
- 오늘 만든 개선·최적화는 **저녁 팀 프로젝트(내 MVP)**에 그대로 적용하세요. 다음 차시(23): 배운 모든 것을 합치는 **캡스톤(업무 모니터링 에이전트)**.